# **CÀI LẠI MÔI TRƯỜNG**

In [ ]:
#Kiểm tra GPU
import sys
import platform
import subprocess

print("Hệ điều hành:", platform.platform())
print("Python:", sys.version)
print("\nThông tin GPU:")

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)


In [ ]:
#Tạo môi trường Python 3.8
%%bash
#Tạo môi trường Python 3.8

set -euo pipefail

mkdir -p /content/hr17_tools

if [ ! -x /content/hr17_tools/bin/micromamba ]; then
    curl -fLsS \
      https://micro.mamba.pm/api/micromamba/linux-64/latest \
      -o /content/hr17_tools/micromamba.tar.bz2

    tar -xjf /content/hr17_tools/micromamba.tar.bz2 \
      -C /content/hr17_tools bin/micromamba
fi

export MAMBA_ROOT_PREFIX=/content/hr17_mamba

if [ ! -x /content/hr17_env/bin/python ]; then
    /content/hr17_tools/bin/micromamba create -y \
      -p /content/hr17_env \
      --override-channels -c conda-forge \
      python=3.8 pip
fi

/content/hr17_env/bin/python --version


In [ ]:
%%bash
#Cài PyTorch theo phiên bản trong log cũ

set -euo pipefail

/content/hr17_env/bin/python -m pip install \
  "pip==24.2" "numpy==1.23.0"

/content/hr17_env/bin/python -m pip install \
  torch==2.4.1 torchvision==0.19.1 \
  --index-url https://download.pytorch.org/whl/cu121


In [ ]:
%%bash
#Kiểm tra tính toán trên CPU
set -euo pipefail

/content/hr17_env/bin/python - <<'PY'
import sys
import numpy as np
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA của PyTorch:", torch.version.cuda)
print("Nhận GPU:", torch.cuda.is_available())

assert torch.cuda.is_available(), "PyTorch chưa nhận GPU"

print("Tên GPU:", torch.cuda.get_device_name(0))

x = torch.ones((100, 100), device="cuda")
y = x @ x
torch.cuda.synchronize()

print("Kết quả kiểm tra:", y[0, 0].item())
print("ĐÃ KIỂM TRA PYTORCH TRÊN GPU THÀNH CÔNG")
PY


In [ ]:
from google.colab import files
from pathlib import Path
import re

names = ['IsaacGym_Preview_4_Package.tar', '20261118_212729_418901.zip',
         '20261118_215810_217119.zip', '20261118_221112_587011.zip']
for name in names:
    target = Path('/content') / name
    if target.is_file():
        print('Đã có:', target)
        continue
    print('Hãy chọn:', name)
    uploaded = files.upload()
    matches = [key for key in uploaded
               if re.sub(r' \(\d+\)(?=\.[^.]+$)', '', Path(key).name) == name]
    assert len(matches) == 1, 'Chọn đúng file: ' + name
    target.write_bytes(uploaded[matches[0]])
    print('Đã lưu:', target, '|', target.stat().st_size, 'bytes')


In [ ]:
#Kiểm tra và giải nén
from pathlib import Path
import tarfile

archive = Path("/content/IsaacGym_Preview_4_Package.tar")

assert archive.exists(), "Chưa thấy file. Hãy kiểm tra tải lên đã hoàn tất."

print("Dung lượng:", archive.stat().st_size, "bytes")
assert archive.stat().st_size == 201141673, (
    "Dung lượng khác file đã kiểm tra trên máy. Hãy kiểm tra lại."
)

with tarfile.open(archive, "r:*") as package:
    package.extractall("/content", filter="data")

assert Path("/content/isaacgym/python/setup.py").exists()
assert Path(
    "/content/isaacgym/python/isaacgym/_bindings/linux-x86_64/gym_38.so"
).exists()

print("ĐÃ GIẢI NÉN ISAAC GYM PREVIEW 4 THÀNH CÔNG")


In [ ]:
%%bash
#Tải repo và kiểm tra nạp robot h1

set -euo pipefail

if [ ! -d /content/hr17_repo ]; then
    git clone https://github.com/LeCAR-Lab/human2humanoid.git \
      /content/hr17_repo
fi

git -C /content/hr17_repo checkout --detach 750f1fa

echo "Commit đang dùng:"
git -C /content/hr17_repo rev-parse --short HEAD

test -f /content/hr17_repo/legged_gym/resources/robots/h1/urdf/h1.urdf

echo "ĐÃ TÌM THẤY MÔ HÌNH H1"


In [ ]:
%%writefile /content/hr17_restore_latest_session.py
ARCHIVE_HASHES = {'20261118_212729_418901.zip': '702e04e7fdba317adf7cd2be6a1863baa3971e32bf3e92991f73cbd10b448a00', '20261118_215810_217119.zip': 'd170dfaf78cc0fb9beb12436a902793aabd3f5c187244dd35556cb7aa98d7448', '20261118_221112_587011.zip': 'ab86848d750f679aa672d63b40709e0512e622ccaeef8190813a1a9e4fef3589'}
"""Restore verified saved experiments and their environment; never train or evaluate."""
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import stat
import subprocess
import sys
import zipfile

CONTENT = Path('/content')
PYTHON = CONTENT / 'hr17_env/bin/python'

def sha(data): return hashlib.sha256(data).hexdigest()

def command(args, cwd=None, capture=False):
    result = subprocess.run(args, cwd=cwd, check=True, text=True,
                            stdout=subprocess.PIPE if capture else None,
                            stderr=subprocess.STDOUT if capture else None)
    return result.stdout if capture else None

def write_same_or_new(destination, data):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.is_file() and destination.read_bytes() == data, 'Existing file differs: ' + str(destination)
    else:
        destination.write_bytes(data)

def verified_zip(name):
    path = CONTENT / name
    assert path.is_file(), 'Upload: ' + name
    assert sha(path.read_bytes()) == ARCHIVE_HASHES[name], 'ZIP differs from verified original: ' + name
    z = zipfile.ZipFile(path)
    assert z.testzip() is None
    manifest = json.loads(z.read('SHA256_manifest.json'))
    for filename, digest in manifest.items():
        assert sha(z.read(filename)) == digest, filename
    return z

def unpack_member(z, name, root, relative=None):
    member = z.getinfo(name)
    assert not stat.S_ISLNK(member.external_attr >> 16), 'Symlink in ZIP'
    relative = name if relative is None else relative
    target = (root / relative).resolve()
    assert os.path.commonpath([str(root.resolve()), str(target)]) == str(root.resolve())
    if not member.is_dir(): write_same_or_new(target, z.read(name))

def ensure_git(path, url, commit, patch):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists(): command(['git', 'clone', url, str(path)])
    assert (path / '.git').exists(), 'Expected git repository: ' + str(path)
    current = command(['git', 'rev-parse', 'HEAD'], cwd=str(path), capture=True).strip()
    if current != commit:
        assert not command(['git', 'status', '--porcelain'], cwd=str(path), capture=True).strip(), 'Existing repository has changes'
        command(['git', 'fetch', 'origin', commit], cwd=str(path))
        command(['git', 'checkout', '--detach', commit], cwd=str(path))
    current_diff = command(['git', 'diff', '--no-ext-diff'], cwd=str(path), capture=True)
    if current_diff.splitlines() != patch.splitlines():
        assert not current_diff.strip(), 'Existing repository patch differs: ' + str(path)
        if patch.strip():
            patch_file = CONTENT / ('hr17_restore_' + path.name + '.patch')
            patch_file.write_text(patch, encoding='utf-8')
            command(['git', 'apply', '--check', str(patch_file)], cwd=str(path))
            command(['git', 'apply', str(patch_file)], cwd=str(path))
    assert command(['git', 'rev-parse', 'HEAD'], cwd=str(path), capture=True).strip() == commit
    assert command(['git', 'diff', '--no-ext-diff'], cwd=str(path), capture=True).splitlines() == patch.splitlines()

def main():
    assert sys.version_info[:2] == (3, 8), 'Run with /content/hr17_env/bin/python'
    assert Path('/content/isaacgym/python/setup.py').is_file(), 'Extract Isaac Gym first'
    # Verify all input files before installing or restoring anything.
    packages = {name: verified_zip(name) for name in ARCHIVE_HASHES}
    training = packages['20261118_212729_418901.zip']
    training_status = json.loads(training.read('audit/experiment_status.json'))
    assert training_status['status'] == 'completed'
    baseline = 'runs/H2O_seed1_20261116_005911/'
    metadata = json.loads(training.read(baseline + 'run_metadata.json'))
    ensure_git(CONTENT / 'hr17_repo', 'https://github.com/LeCAR-Lab/human2humanoid.git',
               metadata['repo_commit'], training.read(baseline + 'repo_changes.patch').decode())
    for name in ['SMPLSim', 'smplx']:
        ensure_git(CONTENT / 'hr17_deps' / name, 'https://github.com/ZhengyiLuo/' + name + '.git',
                   training.read(baseline + name + '_commit.txt').decode().strip(),
                   training.read(baseline + name + '_changes.patch').decode())
    freeze = training.read(baseline + 'environment_freeze.txt').decode()
    pinned = [line.strip() for line in freeze.splitlines()
              if re.fullmatch(r'[A-Za-z0-9_.-]+==[^\s]+', line.strip())]
    requirements = CONTENT / 'hr17_saved_requirements.txt'
    requirements.write_text('\n'.join(pinned) + '\n')
    print('RESTORING SAVED PACKAGE VERSIONS:', len(pinned), flush=True)
    command([str(PYTHON), '-m', 'pip', 'install', '--no-deps',
             '--extra-index-url', 'https://download.pytorch.org/whl/cu121', '-r', str(requirements)])
    editable = [CONTENT / 'isaacgym/python', CONTENT / 'hr17_repo/rsl_rl',
                CONTENT / 'hr17_repo/legged_gym', CONTENT / 'hr17_repo/phc',
                CONTENT / 'hr17_deps/SMPLSim', CONTENT / 'hr17_deps/smplx']
    command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '--no-build-isolation']
            + [part for directory in editable for part in ['-e', str(directory)]])
    actual = command([str(PYTHON), '-m', 'pip', 'freeze'], capture=True)
    lines = lambda s: {line.strip() for line in s.splitlines() if line.strip() and not line.startswith('#')}
    missing, extra = sorted(lines(freeze)-lines(actual)), sorted(lines(actual)-lines(freeze))
    (CONTENT / 'hr17_restore_environment_audit.json').write_text(json.dumps(dict(
        matches_saved_freeze=not missing and not extra, missing_or_different=missing, extra_or_different=extra), indent=2))
    assert not missing and not extra, 'Environment differs; see /content/hr17_restore_environment_audit.json: ' + str((missing, extra))
    group = CONTENT / 'hr17_three_seeds/20261118_212729_418901'
    for name in training.namelist():
        if name.startswith('runs/'):
            unpack_member(training, name, CONTENT / 'hr17_training', name[len('runs/'):])
        elif name.startswith('audit/'):
            unpack_member(training, name, group, name[len('audit/'):])
    for seed, path in training_status['runs'].items():
        folder = Path(path)
        assert folder.parent == CONTENT / 'hr17_training'
        meta = json.loads((folder / 'run_metadata.json').read_text())
        assert meta['seed'] == int(seed) and meta['status'] == 'completed'
        assert sha((folder / 'model_200.pt').read_bytes()) == meta['final_checkpoint_sha256']
    # Pointers require the ZIP alongside its extracted directory.
    shutil.copy2(CONTENT / '20261118_212729_418901.zip', group.with_suffix('.zip'))
    (group.parent / 'latest_archive.txt').write_text(str(group.with_suffix('.zip')))
    for filename, root in [('20261118_215810_217119.zip', 'hr17_three_seed_evaluation'),
                           ('20261118_221112_587011.zip', 'hr17_reward_diagnosis')]:
        z = packages[filename]
        folder = CONTENT / root / Path(filename).stem
        assert json.loads(z.read('experiment_status.json'))['status'] == 'completed'
        for name in z.namelist(): unpack_member(z, name, folder)
        shutil.copy2(CONTENT / filename, folder.with_suffix('.zip'))
        (folder.parent / 'latest_archive.txt').write_text(str(folder.with_suffix('.zip')))
    for filename, archive in [('hr17_evaluate_three_seeds.py', '20261118_215810_217119.zip'),
                               ('hr17_diagnose_rewards.py', '20261118_221112_587011.zip')]:
        write_same_or_new(CONTENT / filename, packages[archive].read(filename))
    baseline_dir = CONTENT / 'hr17_training/H2O_seed1_20261116_005911'
    (baseline_dir.parent / 'latest_run.txt').write_text(str(baseline_dir))
    config_dir = CONTENT / 'hr17_checkpoint_comparison_input'
    for name in ['env_cfg.json', 'train_cfg.json']:
        write_same_or_new(config_dir / name, (baseline_dir / ('source_' + name)).read_bytes())
    env = os.environ.copy()
    env['PATH'] = '/content/hr17_env/bin:' + env.get('PATH', '')
    env['LD_LIBRARY_PATH'] = '/content/hr17_env/lib:/content/isaacgym/python/isaacgym/_bindings/linux-x86_64:' + env.get('LD_LIBRARY_PATH', '')
    env['MAX_JOBS'] = '2'
    check = '''import isaacgym
import torch, numpy as np
from legged_gym.envs import LeggedRobot
assert torch.__version__ == "2.4.1+cu121" and np.__version__ == "1.23.0"
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
print("IMPORT MOI TRUONG H2O THANH CONG")
'''
    subprocess.run([str(PYTHON), '-u', '-c', check], cwd='/content/hr17_repo', env=env, check=True)
    for z in packages.values(): z.close()
    print('KHOI PHUC PHIEN MOI THANH CONG — 3 SEED + DANH GIA + CHAN DOAN', flush=True)
    print('NO TRAINING OR EVALUATION WAS RUN', flush=True)

if __name__ == '__main__': main()


In [ ]:
from google.colab import files
files.download("/content/hr17_driver_diagnosis_v2.zip")

In [ ]:
from google.colab import files
from pathlib import Path
import re
import runpy

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_native_backtrace(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_native_backtrace.py"

script = Path("/content/hr17_native_backtrace.py")
script.write_bytes(uploaded[matches[0]])

runpy.run_path(str(script), run_name="__main__")

In [ ]:
from google.colab import files
from pathlib import Path

pointer = Path("/content/hr17_native_debug/latest_archive.txt")
assert pointer.is_file(), "Hãy gửi output ô vừa chạy nếu chưa có gói log."

archive = Path(pointer.read_text().strip())
files.download(str(archive))

In [ ]:
from pathlib import Path
import os
import json
import subprocess

settings = json.loads(Path(
    "/content/hr17_driver_diagnosis_v2/driver_environment.json"
).read_text())

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = settings["LD_LIBRARY_PATH"]
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["MAX_JOBS"] = "2"
env["PYTHONFAULTHANDLER"] = "1"

code = """
import isaacgym
import torch
from legged_gym.envs import LeggedRobot
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
print("KIEM TRA H2O TRUC TIEP THANH CONG")
"""

result = subprocess.run(
    ["/content/hr17_env/bin/python", "-u", "-c", code],
    cwd="/content/hr17_repo",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=180,
)

print(result.stdout)
print("Mã kết thúc:", result.returncode)

if result.returncode == 0:
    os.environ["LD_LIBRARY_PATH"] = env["LD_LIBRARY_PATH"]
    Path("/content/hr17_runtime_environment.json").write_text(
        json.dumps({
            "LD_LIBRARY_PATH": env["LD_LIBRARY_PATH"],
            "native_h2o_import_verified": True,
        }, indent=2)
    )
    print("ĐÃ LƯU CẤU HÌNH — chạy tiếp ô 10.")
else:
    print("Gửi output này; chưa chạy tiếp ô 10.")

In [ ]:
#Xác nhận dữ liệu đã sẵn sàng
from pathlib import Path
import json
root = Path('/content/hr17_three_seeds')
group = Path((root / 'latest_archive.txt').read_text().strip()).with_suffix('')
status = json.loads((group / 'experiment_status.json').read_text())
for seed, run in status['runs'].items():
    folder = Path(run)
    for filename in ['model_initial.pt', 'model_50.pt', 'model_200.pt', 'env_cfg.json', 'train_cfg.json']:
        assert (folder / filename).is_file(), str(folder / filename)
    print('Đã khôi phục seed', seed, ':', folder)
for name in ['hr17_three_seed_evaluation', 'hr17_reward_diagnosis']:
    folder = Path((Path('/content') / name / 'latest_archive.txt').read_text().strip()).with_suffix('')
    assert json.loads((folder / 'experiment_status.json').read_text())['status'] == 'completed'
    print('Đã khôi phục:', name)
print('SẴN SÀNG TIẾP TỤC CHẨN ĐOÁN CHÂN VÀ TIẾP XÚC — không cần huấn luyện lại')


# **Chẩn đoán chân, tiếp xúc đất và vận tốc khớp**

In [ ]:
#Tải Script
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_diagnose_feet(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_diagnose_feet.py"

destination = Path("/content/hr17_diagnose_feet.py")
destination.write_bytes(uploaded[matches[0]])
print("ĐÃ LƯU:", destination)

In [ ]:
#Chạy chẩn đoán
from pathlib import Path
import json
import os
import subprocess

script = Path("/content/hr17_diagnose_feet.py")
assert script.is_file(), "Hãy chạy ô tải script trước."

runtime = json.loads(Path(
    "/content/hr17_runtime_environment.json"
).read_text())
assert runtime.get("native_h2o_import_verified"), (
    "Cần hoàn thành kiểm tra H2O trực tiếp trước."
)

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = runtime["LD_LIBRARY_PATH"]
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"

with Path("/content/hr17_feet_console.log").open(
    "w", encoding="utf-8"
) as log:
    process = subprocess.Popen(
        ["/content/hr17_env/bin/python", "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)

**Script kiểm tra sáu checkpoint trước/sau huấn luyện và ghi:**
- Độ cao chân robot so với tham chiếu.
- Lực tiếp xúc đất của hai chân.
- Vận tốc khớp và mức vượt giới hạn mềm.
- Trạng thái tại lúc các hàm phạt được gọi.

**Trên cùng 5 bước đầu, khoảng 0,10 giây:**

| Seed | Sai lệch vận tốc khớp trước → sau huấn luyện |
|---|---:|
| 1 | 2,38 → 3,15 rad/s |
| 2 | 2,50 → 4,11 rad/s |
| 3 | 2,28 → 3,60 rad/s |

↪ **Cả ba đều tăng sai lệch**. Ngay bước đầu, hai khuỷu tay có vận tốc khoảng −20 rad/s, trong khi tham chiếu đang chuyển động rất chậm.


In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

pointer = Path("/content/hr17_feet_diagnosis/latest_archive.txt")
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), str(archive)

print("Tải gói:", archive.name)
files.download(str(archive))

# **Kiểm tra lệnh policy được chuyển thành góc đích PD và mô-men như thế nào**

In [ ]:
#Tải Scirpt
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_diagnose_pd(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_diagnose_pd.py"

script = Path("/content/hr17_diagnose_pd.py")
script.write_bytes(uploaded[matches[0]])
print("ĐÃ LƯU:", script)

In [ ]:
#Chạy chẩn đoán PD
from pathlib import Path
import json
import os
import subprocess

script = Path("/content/hr17_diagnose_pd.py")
assert script.is_file(), "Hãy tải script trước."

runtime = json.loads(Path(
    "/content/hr17_runtime_environment.json"
).read_text())
assert runtime.get("native_h2o_import_verified"), (
    "Cần hoàn thành kiểm tra H2O trực tiếp."
)

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = runtime["LD_LIBRARY_PATH"]
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"

with Path("/content/hr17_pd_console.log").open(
    "w", encoding="utf-8"
) as log:
    process = subprocess.Popen(
        ["/content/hr17_env/bin/python", "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)

| Đại lượng ở bước đầu | Khuỷu trái | Khuỷu phải |
|---|---:|---:|
| Góc khởi tạo/tham chiếu | 1,055 rad | 0,885 rad |
| Góc đích do policy đưa tới PD | 0,034 rad | −0,019 rad |
| Mô-men PD tính toán | −102,16 N·m | −90,41 N·m |
| Mô-men sau giới hạn | −18 N·m | −18 N·m |

Cả hai khuỷu bị giới hạn mô-men ở `4/4` bước mô phỏng nhỏ đầu tiên, trong cả sáu trường hợp. Sau khoảng `0,02` giây, vận tốc khuỷu đạt khoảng `−20 rad/s`.

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

pointer = Path("/content/hr17_pd_diagnosis/latest_archive.txt")
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), str(archive)

print("Tải gói:", archive.name)
files.download(str(archive))

# **Kiểm chứng: nếu góc đích hai khuỷu tay khớp tư thế ban đầu, xung mô-men và vận tốc trong 0,02 giây đầu có giảm không?**

In [ ]:
#Tải Script
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()

matches = [
    name for name in uploaded
    if re.fullmatch(
        r"hr17_test_elbow_first_step(?: \(\d+\))?\.py",
        name
    )
]
assert len(matches) == 1, "Chọn file hr17_test_elbow_first_step.py"

script = Path("/content/hr17_test_elbow_first_step.py")
script.write_bytes(uploaded[matches[0]])
print("ĐÃ LƯU:", script)

In [ ]:
#Chạy phép đối chứng
from pathlib import Path
import json
import os
import subprocess

script = Path("/content/hr17_test_elbow_first_step.py")
assert script.is_file(), "Hãy tải script trước."

runtime = json.loads(Path(
    "/content/hr17_runtime_environment.json"
).read_text())
assert runtime.get("native_h2o_import_verified")

env = os.environ.copy()
env["PATH"] = "/content/hr17_env/bin:" + env.get("PATH", "")
env["LD_LIBRARY_PATH"] = runtime["LD_LIBRARY_PATH"]
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONFAULTHANDLER"] = "1"
env["MAX_JOBS"] = "2"
env["MPLBACKEND"] = "Agg"

with Path("/content/hr17_elbow_console.log").open(
    "w", encoding="utf-8"
) as log:
    process = subprocess.Popen(
        ["/content/hr17_env/bin/python", "-u", str(script)],
        cwd="/content/hr17_repo",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

print("\nMã kết thúc:", return_code)

In [ ]:
#Tải kết quả
from google.colab import files
from pathlib import Path

pointer = Path(
    "/content/hr17_elbow_counterfactual/latest_archive.txt"
)
assert pointer.is_file(), "Chưa tìm thấy gói kết quả."

archive = Path(pointer.read_text().strip())
assert archive.is_file(), str(archive)

print("Tải gói:", archive.name)
files.download(str(archive))